## Perform SHAP analysis and generate figures & tables

#### Generate legends

In [1]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib import font_manager as fm
import matplotlib as mpl
from pathlib import Path

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = prop.get_name()

out_dir_legend = Path("../results/figures/shap/legend")
out_dir_legend.mkdir(parents=True, exist_ok=True)

type_colors = {
    "Variant effect predictor": "#BD2424",
    "Mutation-level": "#2ca02c",
    "Gene-level": "#9467bd",
    "Conservation score-based": "#1f77b4",
    "Residue-level": "#c2d41e"
}

legend_patches = [
    Patch(facecolor=color, edgecolor="black", linewidth=0.4, label=label)
    for label, color in type_colors.items()
]

def save_shap_legend():
    fig = plt.figure(figsize=(6, 0.3))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")

    ax.legend(handles=legend_patches,
              loc="center",
              frameon=False,
              ncol=len(type_colors),
              fontsize=14,
              prop=prop)

    out_path = out_dir_legend / "shap_legend_by_type.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(f"Legend saved to: {out_path}")

save_shap_legend()


Legend saved to: ../results/figures/shap/legend/shap_legend_by_type.png


#### Generate importance and summary plots

In [1]:
import os
import re
from pathlib import Path
import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm
import warnings
from tqdm import tqdm 

font_path = "../resources/fonts/InterVariable.ttf"
fm.fontManager.addfont(font_path)
mpl.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()

FONT_SCALE_BAR = 0.9
FONT_SCALE_SUM = 1.0

# Importance plot font sizes
TITLE_SIZE_BAR  = int(17 * FONT_SCALE_BAR)
XLABEL_SIZE_BAR = int(14 * FONT_SCALE_BAR)
XTICK_SIZE_BAR  = int(14 * FONT_SCALE_BAR)
YTICK_SIZE_BAR  = int(12 * FONT_SCALE_BAR)

# Summary plot font sizes
TITLE_SIZE_SUM  = int(17 * FONT_SCALE_SUM)
XLABEL_SIZE_SUM = int(14 * FONT_SCALE_SUM)
XTICK_SIZE_SUM  = int(14 * FONT_SCALE_SUM)
YTICK_SIZE_SUM  = int(12 * FONT_SCALE_SUM)

TITLE_WEIGHT  = "normal"
XLABEL_WEIGHT = "normal"
TICK_WEIGHT   = "normal"

models = [
    "FuncVEP_CTI",
    "FuncVEP_CTE",
    "FuncVEP_SP",
    "ClinVEP_CTI",
    "ClinVEP_CTE",
    "ClinVEP_SP",
]

FEATURE_MATRIX_PATH = "../data/intermediate/feature_matrix_imputed.parquet"
VARIANT_LABELS_PATH = "../data/intermediate/variant_labels.txt"

meta_path  = "../resources/feature_lists/all_columns.txt"
id_column  = "ID"
ensg_col   = "ensg"
target_col = "clinical_testing_label"

max_display = 20
linewidth   = 0.4

out_bar = Path("../results/figures/shap/importance")
out_sum = Path("../results/figures/shap/summary")
out_bar.mkdir(parents=True, exist_ok=True)
out_sum.mkdir(parents=True, exist_ok=True)

meta_df = pd.read_csv(meta_path, sep="\t")
feature_type_map = meta_df.set_index("Name")["Type"].to_dict()

type_colors = {
    "Variant Effect Predictor": "#BD2424",
    "Mutation-Level":           "#2ca02c",
    "Gene-Level":               "#9467bd",
    "Conservation Score-Based": "#1f77b4",
    "Residue-Level":            "#c2d41e",
}

warnings.filterwarnings("ignore", message="LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray")

def clean_feature_name(feat: str) -> str:
    feat = re.sub(r"^glm_", "", feat)
    feat = re.sub(r"_score$", "", feat)
    return feat.replace("___", "-").replace("__", "-").replace("_", "-")


def load_clinical_testing_input() -> pd.DataFrame:
    feature_matrix = pd.read_parquet(FEATURE_MATRIX_PATH)
    variant_labels = pd.read_csv(VARIANT_LABELS_PATH, sep="\t")

    df = (
        variant_labels[[id_column, ensg_col, target_col]]
        .dropna(subset=[target_col])
        .merge(feature_matrix, how="left", on=[id_column, ensg_col])
    )
    del feature_matrix
    return df


def load_model_and_training_set(model_name: str):
    model_dir = f"../models/{model_name}"
    lgb_model = joblib.load(os.path.join(model_dir, "model.pkl"))
    trained_features = list(lgb_model.feature_name_)

    training_set_path = os.path.join(model_dir, "training_set.txt")
    trained_on = pd.read_csv(training_set_path, sep="\t")

    return lgb_model, trained_features, trained_on


def prepare_shap_dataset(
    df_test_raw: pd.DataFrame,
    trained_on: pd.DataFrame,
    trained_features,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    df_test = df_test_raw.copy()
    df_test.columns = df_test.columns.str.replace(" ", "_")

    df_test[id_column] = df_test[id_column].astype(str)
    if ensg_col in df_test.columns:
        df_test[ensg_col] = df_test[ensg_col].astype(str)

    trained_on[id_column] = trained_on[id_column].astype(str)
    if ensg_col in trained_on.columns and ensg_col in df_test.columns:
        trained_on[ensg_col] = trained_on[ensg_col].astype(str)
        train_idx = pd.MultiIndex.from_frame(trained_on[[id_column, ensg_col]])
        test_idx = pd.MultiIndex.from_frame(df_test[[id_column, ensg_col]])
        keep_mask = ~test_idx.isin(train_idx)
        df_test = df_test.loc[keep_mask].copy()
    else:
        df_test = df_test[~df_test[id_column].isin(trained_on[id_column])].copy()

    df_test = df_test.dropna(subset=[target_col])
    df_test[target_col] = df_test[target_col].map({"P": 1, "B": 0})
    df_test = df_test.dropna(subset=[target_col])
    df_test[target_col] = df_test[target_col].astype(int)

    X = df_test[trained_features].copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    y = df_test[target_col].values
    return df_test, X, y


def plot_shap_importance(
    model_name: str,
    X: pd.DataFrame,
    shap_vals: np.ndarray,
):
    shap_means = np.abs(shap_vals).mean(axis=0)
    n_features = shap_means.shape[0]
    n_top = min(max_display, n_features)

    top_idx    = np.argsort(shap_means)[::-1][:n_top]
    top_feats  = X.columns[top_idx]
    top_import = shap_means[top_idx]

    top_colors = [
        type_colors.get(feature_type_map.get(f, "Other"), "#7f7f7f")
        for f in top_feats
    ]
    top_labels = [clean_feature_name(f) for f in top_feats]

    # Choose fig size so axes area is ~square for the chosen margins
    fig_height = 5.0
    left, right, top, bottom = 0.40, 0.98, 0.90, 0.12
    width_frac = right - left
    height_frac = top - bottom
    fig_width = fig_height * (height_frac / width_frac)  # ≈ 6.7

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    positions = np.arange(n_top)[::-1]

    ax.barh(
        positions,
        top_import,
        color=top_colors,
        edgecolor="black",
        linewidth=linewidth,
    )
    ax.set_yticks(positions)
    ax.set_yticklabels(top_labels, fontsize=YTICK_SIZE_BAR, fontweight=TICK_WEIGHT)
    ax.set_xlabel(
        "Mean |SHAP value|",
        fontsize=XLABEL_SIZE_BAR,
        fontweight=XLABEL_WEIGHT,
    )
    ax.set_title(
        model_name.replace("_", "-"),
        fontsize=TITLE_SIZE_BAR,
        fontweight=TITLE_WEIGHT,
        pad=15,
    )
    ax.tick_params(axis="x", labelsize=XTICK_SIZE_BAR)

    # Fixed axes rectangle: consistent proportions across models
    fig.subplots_adjust(
        left=left,
        right=right,
        top=top,
        bottom=bottom,
    )

    # No tight_layout, no bbox_inches="tight"
    fig.savefig(out_bar / f"shap_bar_{model_name}.png", dpi=300)
    plt.close(fig)



def plot_shap_summary(
    model_name: str,
    X: pd.DataFrame,
    shap_vals: np.ndarray,
):
    shap.summary_plot(
        shap_vals,
        X,
        max_display=max_display,
        show=False,
    )
    fig = plt.gcf()
    if fig._suptitle is not None:
        fig._suptitle.set_text("")

    fig.suptitle(
        model_name.replace("_", "-"),
        fontsize=TITLE_SIZE_SUM,
        fontweight=TITLE_WEIGHT,
        y=0.98,
    )

    ax_sum = fig.axes[0]
    ax_sum.tick_params(axis="x", labelsize=XTICK_SIZE_SUM)
    ax_sum.tick_params(axis="y", labelsize=YTICK_SIZE_SUM)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(out_sum / f"shap_summary_{model_name}.png", dpi=600, bbox_inches="tight")
    plt.close()



def run_shap_clinical_plots():
    clinical_testing_input = load_clinical_testing_input()

    for model_name in tqdm(models, desc="SHAP plots", unit="model"):
        lgb_model, trained_features, trained_on = load_model_and_training_set(model_name)
        df_eval, X_eval, y_eval = prepare_shap_dataset(
            clinical_testing_input,
            trained_on,
            trained_features,
        )

        explainer = shap.TreeExplainer(lgb_model)
        shap_vals_raw = explainer.shap_values(X_eval)

        if isinstance(shap_vals_raw, list) and len(shap_vals_raw) == 2:
            shap_vals = shap_vals_raw[1]
        else:
            shap_vals = shap_vals_raw

        plot_shap_importance(model_name, X_eval, shap_vals)
        plot_shap_summary(model_name, X_eval, shap_vals)


run_shap_clinical_plots()

SHAP plots: 100%|██████████| 6/6 [00:25<00:00,  4.25s/model]


#### Combine importance plots

In [2]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
from pathlib import Path
import string
import math

models = ["FuncVEP_CTI", "FuncVEP_CTE", "FuncVEP_SP", "ClinVEP_CTI", "ClinVEP_CTE", "ClinVEP_SP"]
fig_dir_imp = Path("../results/figures/shap/importance")
fig_dir_sum = Path("../results/figures/shap/summary")
legend_path = Path("../results/figures/shap/legend/shap_legend_by_type.png")

out_imp = fig_dir_imp / "shap_bar_combined.png"
out_sum = fig_dir_sum / "shap_summary_combined.png"

func_models = ["FuncVEP_CTI", "FuncVEP_CTE", "FuncVEP_SP"]
clin_models = ["ClinVEP_CTI", "ClinVEP_CTE", "ClinVEP_SP"]

imp_paths = []
for f, c in zip(func_models, clin_models):
    imp_paths.append(fig_dir_imp / f"shap_bar_{f}.png")
    imp_paths.append(fig_dir_imp / f"shap_bar_{c}.png")

def build_grid_two_columns(img_paths, out_path, include_legend=False, legend_img=None,
                           wspace=0.2, hspace=0.2, figsize_override=None):
    if include_legend and legend_img is None:
        raise ValueError("legend_img path must be provided when include_legend=True")

    n = len(img_paths)
    cols = 2
    rows = math.ceil(n / cols)
    height_ratios = [1] * rows

    if include_legend:
        rows += 1
        height_ratios.append(0.18)

    gs = gridspec.GridSpec(rows, cols, height_ratios=height_ratios)

    if figsize_override is None:
        fig = plt.figure(figsize=(cols * 7, rows * 4))
    else:
        fig = plt.figure(figsize=figsize_override)

    for idx, img_path in enumerate(img_paths):
        row = idx // cols
        col = idx % cols
        ax = fig.add_subplot(gs[row, col])
        img = mpimg.imread(img_path)
        ax.imshow(img, aspect='auto')
        ax.axis("off")

        ax.text(
            0.01,
            0.97,
            f"{string.ascii_uppercase[idx]}",
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            va="top",
            ha="left",
        )

    if include_legend:
        legend_ax = fig.add_subplot(gs[-1, :])
        legend_ax.axis("off")
        legend_img_data = mpimg.imread(legend_img)
        legend_ax.imshow(legend_img_data)
        legend_ax.set_anchor("N")

    plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=wspace, hspace=hspace)
    plt.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved to {out_path.relative_to(out_path.parents[2])}")

build_grid_two_columns(
    imp_paths,
    out_imp,
    include_legend=True,
    legend_img=legend_path,
    wspace=0.2,
    hspace=0.2,
)

saved to shap\importance\shap_bar_combined.png


#### Combine summary plots

In [4]:
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import numpy as np

def collect_shap_importances():
    clinical_testing_input = load_clinical_testing_input()
    importance_dict = {}

    for model_name in models:
        lgb_model, trained_features, trained_on = load_model_and_training_set(model_name)
        df_eval, X_eval, y_eval = prepare_shap_dataset(
            clinical_testing_input,
            trained_on,
            trained_features,
        )

        explainer = shap.TreeExplainer(lgb_model)
        shap_vals_raw = explainer.shap_values(X_eval)

        if isinstance(shap_vals_raw, list) and len(shap_vals_raw) == 2:
            shap_vals = shap_vals_raw[1]
        else:
            shap_vals = shap_vals_raw

        shap_means = np.abs(shap_vals).mean(axis=0)
        importance_series = pd.Series(shap_means, index=X_eval.columns, name=model_name)

        importance_dict[model_name] = importance_series

    importance_df = pd.DataFrame(importance_dict)
    importance_df.index.name = "Feature"
    importance_df.sort_values(by="FuncVEP_CTI", ascending=False, inplace=True)
    importance_df = importance_df.reset_index()

    out_dir = Path("../results/tables/shap")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "all_models_shap_importances.txt"

    importance_df.to_csv(out_path, sep="\t", index=False)
    print("Saved SHAP importance table to:", out_path)

collect_shap_importances()

Saved SHAP importance table to: ../results/tables/shap/all_models_shap_importances.txt


#### Generate importance tables

In [5]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

models = ["FuncVEP_CTI", "ClinVEP_CTI", "FuncVEP_CTE", "ClinVEP_CTE", "FuncVEP_SP", "ClinVEP_SP"]

meta_df = pd.read_csv("../resources/feature_lists/all_columns.txt", sep="\t")
feature_type_map = meta_df.set_index("Name")["Category"].to_dict()

unique_types = sorted(meta_df["Category"].dropna().unique())
if "FuncVEP" in unique_types:
    unique_types.remove("FuncVEP")

summary_by_type = {}

clinical_testing_input = load_clinical_testing_input()

for model_name in models:
    lgb_model, trained_features, trained_on = load_model_and_training_set(model_name)
    
    df_eval, X_eval, y_eval = prepare_shap_dataset(
        clinical_testing_input,
        trained_on,
        trained_features,
    )

    explainer = shap.TreeExplainer(lgb_model)
    shap_vals_full = explainer.shap_values(X_eval)
    if isinstance(shap_vals_full, list) and len(shap_vals_full) == 2:
        shap_vals = shap_vals_full[1]
    else:
        shap_vals = shap_vals_full

    shap_means = np.abs(shap_vals).mean(axis=0)

    importance_df = pd.DataFrame({
        "feature": X_eval.columns,
        "mean_abs_shap": shap_means,
    })
    importance_df["type"] = importance_df["feature"].map(feature_type_map).fillna("Other")

    grouped = importance_df.groupby("type")["mean_abs_shap"]
    total_importance = grouped.sum()
    average_importance = grouped.mean()

    summary_by_type[model_name] = {
        "total": total_importance.to_dict(),
        "average": average_importance.to_dict(),
    }

# Build total and average importance tables by type x model
total_df = pd.DataFrame(index=unique_types, columns=models, dtype=float)
average_df = pd.DataFrame(index=unique_types, columns=models, dtype=float)

for model in models:
    for t in unique_types:
        total_df.loc[t, model] = summary_by_type[model]["total"].get(t, 0.0)
        average_df.loc[t, model] = summary_by_type[model]["average"].get(t, 0.0)

out_dir = Path("../results/tables/shap")
out_dir.mkdir(parents=True, exist_ok=True)

total_path = out_dir / "shap_total_importance_by_type.txt"
average_path = out_dir / "shap_average_importance_by_type.txt"

total_df.to_csv(total_path, index_label="Feature_Type", sep="\t")
average_df.to_csv(average_path, index_label="Feature_Type", sep="\t")

# Normalized versions (each column sums to 1)
total_df_normalized = total_df.div(total_df.sum(axis=0), axis=1)
average_df_normalized = average_df.div(average_df.sum(axis=0), axis=1)

total_norm_path = out_dir / "shap_total_importance_by_type_normalized.txt"
average_norm_path = out_dir / "shap_average_importance_by_type_normalized.txt"

total_df_normalized.to_csv(total_norm_path, index_label="Feature_Type", sep="\t")
average_df_normalized.to_csv(average_norm_path, index_label="Feature_Type", sep="\t")

print("SHAP importance values by type saved.")

SHAP importance values by type saved.
